## Homework 8: Neural Networks and Deep Learning
In this homework we build a small PyTorch model that classifies curly and straight hair. The 2026 reference setup freezes the dataset, framework versions, atchitecture, split, random seed, and evaluation procedure. These settings make the experiment reproducible while still showing why augmentation can change a model's behavior.

### Dataset
Download the fixed archive while using MacOS:

```bash
curl -L -O https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
shasum -a 256 data.zip
unzip -q data.zip
```

The archive contains a fixed training split and a fixed held-out evaluation split. It has 1,002 image files. One GIF is kept in the archive for provenance but ImageFolder ignores it, so the training run consumes 1,001 images:

|Split	|Curly	|Straight	|Total|
|---|---|---|---|
|Train	|410	|390	|800|
|Held-out evaluation	|103	|98	|201|

`torchvision.datasets.ImageFolder` assigns curly=0 and straight=1 because the class directories are sorted alphabetically. Do not create another random train/test split and do not augment the held-out evaluation images.

### Environments (for MacOS)
```bash
uv init --bare --no-workspace --python 3.11
uv python pin 3.11
uv add torch==2.9.0 torchvision==0.24.0 numpy==2.3.3 pillow==11.3.0
uv add --dev jupyter ipykernel
```

The directory also contains `reference_train.py.` After extracting the archive, run `python reference_train.py --data-dir` data to produce both history files with the complete reference procedure.

### Reproducibility rules
Run this before constructing datasets, data loaders, the model, or the optimizer:

```python
import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")
```

Use `num_workers = 0` for both loaders. Give the training loader its own fixed generator:
```python
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)
```
These settings remove the worker-order and GPU/cuDNN differences that made the 2025 scalar answers move between runs.

### Model and preprocessing
Use this transform for the baseline training set and, separately, for the held-out evaluation set:
```python
from torchvision import transforms

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])
```
Create the model with exactly this structure:
```python
import torch.nn as nn


class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)
```

The model returns logits. Use `nn.BCEWithLogitsLoss()` and apply `torch.sigmoid` only when calculating predictions or serving the model. Do not put a sigmoid module in `HairModel` as well; applying it twice was an ambiguity in the previous assignment.

Use this optimizer and loss:
```python
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)
```

In [22]:
import random
from torchvision import transforms
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image 
from pathlib import Path
import os
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [23]:
class HairDataset(Dataset):
    
    def __init__(self, data_dir, transform = None) -> None:
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {"curly": 0, "straight": 1}
        
        for label in sorted(os.listdir(data_dir)):
            if label not in self.class_to_idx:
                continue
            curr_dir = os.path.join(data_dir, label)
            
            for img_file in sorted(os.listdir(curr_dir)):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.image_paths.append(os.path.join(curr_dir, img_file))
                    self.labels.append(self.class_to_idx[label])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, index):
        img_path = self.image_paths[index]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[index]
        
        if self.transform:
            image = self.transform(image)
        return image, label

In [24]:
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [25]:
train_dataset = HairDataset(
    data_dir= './data/train',
    transform= evaluation_transform,
)

evaluation_dataset = HairDataset(
    data_dir= './data/test',
    transform= evaluation_transform,
)

In [26]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [27]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride = 1, padding = 0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)
        
    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim = 1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

In [28]:
model = HairModel()

In [29]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

### Question 1 -- Loss and output contract
Which loss matches the model above?
- `nn.MSELoss()`
- `nn.BCEWithLogitsLoss()`
- `nn.CrossEntropyLoss()`
- `nn.CosineEmbeddingLoss()`

There is one intended answer: the model emits one logit and the loss applies the stable sigmoid-plus-binary-cross-entropy operation.

**Ans:** `nn.BCEWithlogitsLoss()`. It is because the model returns only one logit for each image.

----

### Question 2 -- parameter count
What is the total number of trainable parameters? Count biases as well and use `sum(p.numel() for p in model.parameters())`.
- 896
- 11214912
- 15896912
- 20073473

In [30]:
for name, p in model.named_parameters():
    print(name, tuple(p.shape), p.numel())

conv.weight (32, 3, 3, 3) 864
conv.bias (32,) 32
hidden.weight (64, 313632) 20072448
hidden.bias (64,) 64
output.weight (1, 64) 64
output.bias (1,) 1


In [31]:
sum(p.numel() for p in model.parameters())

20073473

**Ans:** 20073473.

----

### Training Metrics
Train for 10 epochs with the baseline transform. For each epoch, evaluate the held-out split with `model.eval()` and `torch.no_grad()`. Store these four series in a JSON file named `history_baseline.json`:
```json
{
    "train_loss": [...],
    "train_accuracy": [...],
    "evaluation_loss": [...],
    "evaluation_accuracy": [...],
}
```

For accuracy, use `(torch.sigmoid(logits) >= 0.5)`. Use the number of examples, not the number of batches, when averaging loss and accuracy.

### Question 3 -- baseline accuracy
Calculate the median of `train_accuracy` in `history_baseline.json`. Submit the value rounded to two decimal places. The grader accepts and absolute error of `0.05`; the answer is not selected from a fixed option list.

In [32]:
def train_and_eval(num_epochs, model, optimizer, criterion, train_loader, evaluation_loader, device, history_path = None):
    train_loss = []
    train_accuracy = []
    evaluation_loss = []
    evaluation_accuracy = []
    for epoch in range(num_epochs):
        # Training phase
        model.train() # Set the model to training mode
        running_loss = 0.0
        correct = 0
        total = 0
        
        # Iterate over the training data
        for inputs, labels in train_loader:
            # Move data to the specified device (CPU)
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            # Zero the parameter gradients to prevent accumulation
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            
            # calculate the loss
            loss = criterion(input = outputs, target = labels)
            
            # backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Accumulate training loss
            running_loss += loss.item() * labels.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        # Calculate average training loss and accuracy
        train_epch_loss = running_loss / total
        train_epch_accuracy = correct / total
            
        # Evaluation phase
        model.eval() # Set the model to validation mode
        eval_running_loss = 0.0
        eval_epch_correct = 0
        eval_epch_total = 0
        
        # Disable gradient calculation for validation
        with torch.no_grad():
            # Iterate over the evaluation data
            for inputs, labels in evaluation_loader:
                # Move data to specified device (CPU)
                inputs, labels = inputs.to(device), labels.to(device)
                labels = labels.float().unsqueeze(1)
                # Forward pass
                outputs = model(inputs)
                # Calculate the loss
                loss = criterion(input = outputs, target = labels)
                
                eval_running_loss += loss.item() * labels.size(0)
                eval_epch_total += labels.size(0)
                eval_epch_preds = (torch.sigmoid(outputs) >= 0.5).float()
                eval_epch_correct += (eval_epch_preds == labels).sum().item()
        eval_epch_loss = eval_running_loss / eval_epch_total
        eval_epch_accuracy = eval_epch_correct / eval_epch_total
        
        train_loss.append(train_epch_loss)
        train_accuracy.append(train_epch_accuracy)
        evaluation_loss.append(eval_epch_loss)
        evaluation_accuracy.append(eval_epch_accuracy)
        
    history = {
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'evaluation_loss': evaluation_loss,
        'evaluation_accuracy': evaluation_accuracy,
    }
    if history_path:
        with open(f'{history_path}.json', 'w', encoding='utf-8') as f:
            json.dump(history, f, indent=2)
            print(f'{history_path}.json saved')
    return history


In [33]:
my_train_history = train_and_eval(num_epochs=10, model=model,
                                optimizer=optimizer, criterion= criterion,
                                train_loader=train_loader, evaluation_loader=evaluation_loader,
                                device= device, history_path='my_history_baseline')

my_history_baseline.json saved


In [34]:
median_train_accuracy = np.median(my_train_history['train_accuracy']).round(2)
print(median_train_accuracy)

0.81


In [35]:
# Checking the ans from reference_train.py
ref = json.load(open('runs/reference/history_baseline.json'))
print(np.median(ref["train_accuracy"]).round(2))

0.81


**Ans:** The median of train_accuracy is `0.81`

----

### Question 4 -- baseline loss
Calculate the population standard deviation of `train_loss` in `history_baseline.json` (for NumPy, use `np.std(values, ddof=0))`. Submit the value rounded to three decimal places. The grader accepts an absolute error of `0.02`.

In [36]:

std_train_loss = np.std(my_train_history['train_loss'], ddof=0).round(3)
print(std_train_loss)


0.139


In [37]:
# Checking the ans from reference_train.py
ref = json.load(open('runs/reference/history_baseline.json'))
print(np.std(ref["train_loss"], ddof=0).round(3))

0.139


**Ans:** The std of train_loss is `0.139`

----

### Data augmentation
Keep the same model and optimizer. Replace only the training transform with:
```python
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])
```

Do not recreate the model or optimizer. Train for 10 additional epochs and write only those ten epochs to history_augmented.json. Continue using the unchanged, deterministic evaluation_dataset and evaluation_loader for every evaluation.

In [38]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [ ]:
loader_generator2 = torch.Generator().manual_seed(SEED+1) # make sure the seed is as same as reference.py
train_dataset = HairDataset(
    data_dir= './data/train',
    transform= train_transform_augmented,
)

train_loader_augmented = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator2,
)

In [40]:
my_aug_train_history = train_and_eval(num_epochs=10, model=model,
                                optimizer=optimizer, criterion= criterion,
                                train_loader=train_loader_augmented, evaluation_loader=evaluation_loader,
                                device= device, history_path='my_history_augmented')

my_history_augmented.json saved


### Question 5: augmented evaluation loss
Calculate the mean of `evaluation_loss` in history_augmented.json. Submit the value rounded to three decimal places. The grader accepts an absolute error of 0.02.

In [41]:
mean_eval_loss = np.mean(my_aug_train_history['evaluation_loss']).round(3)
print(mean_eval_loss)

0.575


In [42]:
# checking the mean evaluation loss from reference
ref_aug = json.load(open('runs/reference/history_augmented.json'))
print(np.mean(ref_aug["evaluation_loss"]).round(3))

0.575


**Ans:** The mean of augmented evaluation_loss is 0.575.

----

### Question 6 -- augmented evaluation accuracy
Calculate the mean of the last five values in evaluation_accuracy in history_augmented.json (epochs 6 through 10 of the augmented phase). Submit the value rounded to two decimal places. The grader accepts an absolute error of 0.05.

The tolerances cover minor CPU wheel differences, while the fixed seed, loader, split, architecture, and inference transform prevent the large uncontrolled variation from the 2025 exercise.

In [44]:
last_5 = my_aug_train_history['evaluation_accuracy'][-5:]
print(np.mean(last_5).round(2))

0.72


In [45]:
# checking the mean evaluation loss from reference
ref_aug = json.load(open('runs/reference/history_augmented.json'))
print(np.mean(ref_aug["evaluation_accuracy"][-5:]).round(2))

0.72


**Ans:** The mean of the last five values in augmented evaluation_accuracy is 0.72.

----
End of this page